# Notebook 01  —  Label Generation (6 Classes)

**Class mapping:**

| ID | Class | 
|----|-------|
| 0  | Maize | 
| 1  | Maize+Pumpkin | 
| 2  | Beans+Maize | 
| 3  | Cassava+Maize |
| 4  | Grass |
| 5  | Mixed |
| 255 | Background / No-data |

## 0. Imports and paths

In [5]:
import ast
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR     = Path(r"E:\THESIS_COLLINS HLORDZIE\02_PROCESSED")
SA_ROOT      = BASE_DIR
ENUM_CSV     = BASE_DIR / "Enumerator_Data" / "Enumerator_Data.xls"
SEGMENTS_SHP = BASE_DIR / "Enumerator_Data" / "Enumerator_Data.shp"
OUTPUT_DIR   = BASE_DIR / "Labels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SKIP_SAS = {"SA012_10445982"}

# ── Column names ───────────────────────────────────────────────────────────
COL_PLOTID     = "newgid"
COL_CROPLIST   = "croplist_e"
COL_CROP_COV   = "crop_cov_1"
COL_DOMINANT_G = "dominant_g"
COL_SA_TASK    = "sa_task"

# ── Class encoding (6 classes) ─────────────────────────────────────────────
CLASS_MAP = {
    "Maize":         0,
    "Maize+Pumpkin": 1,
    "Beans+Maize":   2,
    "Cassava+Maize": 3,
    "Grass":         4,
    "Mixed":         5,
}
CLASS_NAMES = {v: k for k, v in CLASS_MAP.items()}
BACKGROUND  = 255

print("Config loaded. Classes:")
for cid, cname in CLASS_NAMES.items():
    print(f"  {cid} = {cname}")

Config loaded. Classes:
  0 = Maize
  1 = Maize+Pumpkin
  2 = Beans+Maize
  3 = Cassava+Maize
  4 = Grass
  5 = Mixed


## 1. Load CSV and audit unique crop names

In [7]:
# ── Load the enumerator data (XLS/XLSX format)
import os

# Auto-detect the actual file extension
enum_dir = BASE_DIR / "Enumerator_Data"
xls_candidates = list(enum_dir.glob("Enumerator_Data.*"))
print("Files found in Enumerator_Data folder:")
for f in xls_candidates:
    print(f"  {f.name}")

# Pick the spreadsheet file (xls or xlsx)
spreadsheet = next(
    (f for f in xls_candidates if f.suffix.lower() in (".xls", ".xlsx")), None
)

if spreadsheet is None:
    raise FileNotFoundError(
        "No .xls or .xlsx file found in the Enumerator_Data folder. "
        "Check the path or file name."
    )

print(f"\nReading: {spreadsheet.name}")
df = pd.read_excel(
    spreadsheet,
    engine="openpyxl" if spreadsheet.suffix == ".xlsx" else "xlrd"
)

print(f"Loaded: {len(df)} rows, {df.shape[1]} columns\n")
print("Columns:", df.columns.tolist())

Files found in Enumerator_Data folder:
  Enumerator_Data.cpg
  Enumerator_Data.dbf
  Enumerator_Data.prj
  Enumerator_Data.sbn
  Enumerator_Data.sbx
  Enumerator_Data.shp
  Enumerator_Data.shp.xml
  Enumerator_Data.shx
  Enumerator_Data.xls

Reading: Enumerator_Data.xls
Loaded: 1205 rows, 41 columns

Columns: ['OBJECTID', 'province', 'sa_task', 'plotid', 'newgid', 'segment_ta', 'landform_e', 'profile', 'farming_ty', 'optional', 'is_visited', '_uid_', 'id', 'formid', 'report_clo', 'enumerator', 'lulc1', 'lulc2', 'is_cultiva', 'tasked', 'areasize_w', 'prepared_l', 'landcover', 'prepared_1', 'landuse', 'size_in_sa', 'dominant_g', 'croplist_e', 'crop_cov_e', 'crop_cov_1', 'tree_canop', 'nr_small_t', 'nr_average', 'nr_large_t', 'noncrop_en', 'noncrop_co', 'image1', 'image2', 'image3', 'image4', 'image5']


In [8]:
def parse_croplist(raw):
    if pd.isna(raw) or str(raw).strip() in ("", "[]", "nan"):
        return frozenset()
    try:
        parsed = ast.literal_eval(str(raw))
        if isinstance(parsed, list):
            return frozenset(c.strip().lower() for c in parsed if c)
    except (ValueError, SyntaxError):
        pass
    cleaned = re.sub(r"[\[\]']", "", str(raw))
    items = [c.strip().lower() for c in cleaned.split(",") if c.strip()]
    return frozenset(items)

df["_crops"] = df[COL_CROPLIST].apply(parse_croplist)

# Audit unique crop names
all_crops = Counter()
for crops in df["_crops"]:
    for c in crops:
        all_crops[c] += 1

print("=" * 50)
print(f"Unique crop names ({len(all_crops)} total):")
print("=" * 50)
for crop, count in all_crops.most_common():
    print(f"  {crop:30s}  {count:4d} segments")

# Combination frequencies
combo_counts = Counter()
for crops in df["_crops"]:
    key = "+".join(sorted(crops)) if crops else "(empty)"
    combo_counts[key] += 1

print("\n" + "=" * 50)
print("Crop combinations:")
print("=" * 50)
for combo, count in combo_counts.most_common():
    flag = "  <- < 8 segments (-> Mixed)" if count < 8 else ""
    print(f"  {combo:40s}  {count:4d}{flag}")

Unique crop names (15 total):
  maize                            240 segments
  beans                             59 segments
  pumpkin                           37 segments
  cassava                           29 segments
  other vegetables                  24 segments
  sorghum                           22 segments
  groundnut (large)                  9 segments
  okra                               7 segments
  millet                             6 segments
  groundnut (small)                  6 segments
  sugarcane                          6 segments
  sweet potato (non-orange flesh)     5 segments
  rice                               4 segments
  yam                                2 segments
  onion                              2 segments

Crop combinations:
  (empty)                                    936
  maize                                      129
  maize+pumpkin                               21
  beans+maize                                 18
  cassava+maize                  

## 2. Assign class labels

Priority order:
1. **Grass** — grass-only dominant_g AND (no crops OR crop_cov == 0)
2. **Maize** — only maize AND grass in dominant_g (former Maize+Grass → now Maize)
3. **Known pure/combo classes** (>= 8 segments, <= 2 crops)
4. **Background (255)** — empty segments with no crops and no grass signal
5. **Mixed** — rare combos, 3+ crop combos

In [9]:
def parse_dominant_g(raw):
    if pd.isna(raw) or str(raw).strip() in ("", " ", "[]", "nan"):
        return frozenset()
    try:
        parsed = ast.literal_eval(str(raw).strip())
        if isinstance(parsed, list):
            return frozenset(v.strip().lower() for v in parsed if v)
    except (ValueError, SyntaxError):
        pass
    cleaned = re.sub(r"[\[\]']", "", str(raw))
    items = [v.strip().lower() for v in cleaned.split(",") if v.strip()]
    return frozenset(items)


# Qualifying combinations: pure or two-crop, >= 8 segments
QUALIFYING_COMBOS = {
    combo for combo, count in combo_counts.items()
    if count >= 8 and combo != "(empty)" and len(combo.split("+")) <= 2
}

NAMED_COMBOS = {
    "maize":         CLASS_MAP["Maize"],
    "maize+pumpkin": CLASS_MAP["Maize+Pumpkin"],
    "beans+maize":   CLASS_MAP["Beans+Maize"],
    "cassava+maize": CLASS_MAP["Cassava+Maize"],
}

print("Qualifying combinations (>= 8 segments, <= 2 crops):")
for c in sorted(QUALIFYING_COMBOS):
    label = NAMED_COMBOS.get(c, CLASS_MAP["Mixed"])
    print(f"  {c:30s} -> class {label}")


def assign_class(crops, dominant_g_set, crop_cov):
    try:
        cov = float(crop_cov)
    except (TypeError, ValueError):
        cov = None

    has_grass  = "grass" in dominant_g_set
    has_woody  = bool(dominant_g_set & {"shrubs", "trees"})
    grass_only = has_grass and not has_woody and dominant_g_set <= {"grass", "other"}
    no_crops   = (len(crops) == 0) or (cov is not None and cov == 0)

    # Rule 1 — Grass (pure grass segment, no crops)
    if grass_only and no_crops:
        return CLASS_MAP["Grass"]

    # Rule 2 — Maize+Grass merged into Maize
    # (only maize reported AND grass in dominant_g AND no woody vegetation)
    if crops == frozenset({"maize"}) and has_grass and not has_woody:
        return CLASS_MAP["Maize"]

    # Rule 3 — Empty segment (no crops, no grass signal) -> Background
    if no_crops:
        return BACKGROUND

    combo   = "+".join(sorted(crops))
    n_crops = len(crops)

    # Rule 4 — Three or more crops -> Mixed
    if n_crops >= 3:
        return CLASS_MAP["Mixed"]

    # Rule 5 — Known named class (>= 8 segments)
    if combo in NAMED_COMBOS and combo in QUALIFYING_COMBOS:
        return NAMED_COMBOS[combo]

    # Rule 6 — Rare combo -> Mixed
    return CLASS_MAP["Mixed"]


df["_dom_g_set"] = df[COL_DOMINANT_G].apply(parse_dominant_g)
df["_class_id"]  = df.apply(
    lambda r: assign_class(r["_crops"], r["_dom_g_set"], r[COL_CROP_COV]), axis=1
)

print("\nClass distribution across all segments:")
dist = df["_class_id"].value_counts().sort_index().reset_index()
dist.columns = ["class_id", "count"]
dist["class_name"] = dist["class_id"].map({**CLASS_NAMES, BACKGROUND: "Background"})
dist["pct"] = (dist["count"] / dist["count"].sum() * 100).round(1)
display(dist[["class_id", "class_name", "count", "pct"]])

Qualifying combinations (>= 8 segments, <= 2 crops):
  beans+maize                    -> class 2
  cassava+maize                  -> class 3
  maize                          -> class 0
  maize+pumpkin                  -> class 1
  maize+sorghum                  -> class 5

Class distribution across all segments:


,class_id,class_name,count,pct
0,0,Maize,129,10.7
1,1,Maize+Pumpkin,21,1.7
2,2,Beans+Maize,18,1.5
3,3,Cassava+Maize,10,0.8
4,4,Grass,175,14.5
5,5,Mixed,91,7.6
6,255,Background,761,63.2


## 3. Load shapefile and join class labels

In [10]:
print("Loading segment shapefile ...", flush=True)
gdf = gpd.read_file(SEGMENTS_SHP)
print(f"Loaded: {len(gdf)} segments, CRS = {gdf.crs}")

# Clean sa_task (strip .0 float suffix)
gdf[COL_SA_TASK] = (
    gdf[COL_SA_TASK].astype(str).str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

# Clean newgid join key
gdf[COL_PLOTID] = (
    gdf[COL_PLOTID].astype(str).str.strip()
    .str.replace(r"\.0$", "", regex=True)
)
df[COL_PLOTID] = (
    df[COL_PLOTID].astype(str).str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

# Sanity check
common = set(gdf[COL_PLOTID]) & set(df[COL_PLOTID])
print(f"Matching newgid values : {len(common)}")
print(f"Shapefile segments     : {len(gdf)}")
print(f"CSV rows               : {len(df)}")

# Join
label_lookup = df[[COL_PLOTID, "_class_id"]].drop_duplicates(subset=COL_PLOTID)
gdf = gdf.merge(label_lookup, on=COL_PLOTID, how="left")
gdf["_class_id"] = gdf["_class_id"].fillna(BACKGROUND).astype(np.uint8)

unmatched = gdf["_class_id"].eq(BACKGROUND).sum()
print(f"Segments labelled      : {len(gdf) - unmatched}")
print(f"Segments unmatched     : {unmatched}")
print("Join complete.")

Loading segment shapefile ...
Loaded: 1205 segments, CRS = EPSG:3036
Matching newgid values : 1205
Shapefile segments     : 1205
CSV rows               : 1205
Segments labelled      : 444
Segments unmatched     : 761
Join complete.


## 4. Helper functions

In [11]:
def get_sa_id(folder_name):
    return folder_name.split("_", 1)[-1]

def find_raster(sa_dir, keyword):
    for f in sa_dir.iterdir():
        if keyword in f.name and f.suffix in (".tif", ".tiff"):
            return f
    return None

print("Helpers defined.")

Helpers defined.


## 5. Generate Labels

In [13]:
sa_folders = sorted([
    d for d in SA_ROOT.iterdir()
    if d.is_dir() and d.name.startswith("SA") and d.name not in SKIP_SAS
])

print(f"Processing {len(sa_folders)} SAs ...\n")
results = []

for sa_dir in sa_folders:
    sa_name  = sa_dir.name
    sa_id    = get_sa_id(sa_name)
    out_path = OUTPUT_DIR / f"{sa_id}_labels.tif"

    ref_raster = find_raster(sa_dir, f"{sa_id}_stack_RGB")
    if ref_raster is None:
        print(f"  [WARN] {sa_name} — RGB raster not found, skipping.", flush=True)
        results.append({"SA": sa_name, "status": "no RGB raster", "valid_px": 0})
        continue

    mask_raster = find_raster(sa_dir, "cropland_mask")
    if mask_raster is None:
        print(f"  [WARN] {sa_name} — vegetation mask not found.", flush=True)

    with rasterio.open(ref_raster) as src:
        transform = src.transform
        crs       = src.crs
        height    = src.height
        width     = src.width

    # Filter segments for this SA using numeric sa_id
    gdf_sa = gdf[gdf[COL_SA_TASK] == sa_id].copy()
    if gdf_sa.empty:
        print(f"  [WARN] {sa_name} — no segments found, skipping.", flush=True)
        results.append({"SA": sa_name, "status": "no segments", "valid_px": 0})
        continue

    if gdf_sa.crs != crs:
        gdf_sa = gdf_sa.to_crs(crs)

    shapes = (
        (geom, int(val))
        for geom, val in zip(gdf_sa.geometry, gdf_sa["_class_id"])
        if geom is not None and not geom.is_empty
    )
    label_arr = rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=BACKGROUND,
        dtype=np.uint8,
        all_touched=False,
    )

    if mask_raster is not None:
        with rasterio.open(mask_raster) as msrc:
            mask_data = msrc.read(1)
        label_arr[mask_data == 0] = BACKGROUND

    with rasterio.open(
        out_path, "w",
        driver="GTiff", dtype="uint8",
        width=width, height=height, count=1,
        crs=crs, transform=transform,
        compress="lzw", nodata=BACKGROUND,
    ) as dst:
        dst.write(label_arr, 1)

    valid_px = int(np.sum(label_arr != BACKGROUND))
    print(f"  [OK] {sa_name} -> {out_path.name} ({valid_px:,} labelled pixels)",
          flush=True)
    results.append({"SA": sa_name, "status": "ok", "valid_px": valid_px})

print("\nDone.")
display(pd.DataFrame(results))

Processing 19 SAs ...

  [OK] SA010_10305790 -> 10305790_labels.tif (59,824,342 labelled pixels)
  [OK] SA011_10445787 -> 10445787_labels.tif (17,767,320 labelled pixels)
  [OK] SA013_10565691 -> 10565691_labels.tif (69,470,233 labelled pixels)
  [OK] SA014_10595716 -> 10595716_labels.tif (69,857,079 labelled pixels)
  [OK] SA015_10605304 -> 10605304_labels.tif (70,637,415 labelled pixels)
  [OK] SA016_10765685 -> 10765685_labels.tif (32,557,625 labelled pixels)
  [OK] SA017_10865406 -> 10865406_labels.tif (39,114,229 labelled pixels)
  [OK] SA018_10865805 -> 10865805_labels.tif (21,766,502 labelled pixels)
  [OK] SA019_11125706 -> 11125706_labels.tif (21,740,921 labelled pixels)
  [OK] SA01_9545448 -> 9545448_labels.tif (53,753,140 labelled pixels)
  [OK] SA020_11226291 -> 11226291_labels.tif (140,650 labelled pixels)
  [OK] SA02_9805753 -> 9805753_labels.tif (7,138,284 labelled pixels)
  [OK] SA03_9965805 -> 9965805_labels.tif (2,363,797 labelled pixels)
  [OK] SA04_10085703 -> 10085

,SA,status,valid_px
0,SA010_10305790,ok,59824342
1,SA011_10445787,ok,17767320
2,SA013_10565691,ok,69470233
3,SA014_10595716,ok,69857079
4,SA015_10605304,ok,70637415
5,SA016_10765685,ok,32557625
6,SA017_10865406,ok,39114229
7,SA018_10865805,ok,21766502
8,SA019_11125706,ok,21740921
9,SA01_9545448,ok,53753140


## 6. Per-SA class distribution check

In [14]:
rows = []
for label_path in sorted(OUTPUT_DIR.glob("*_labels.tif")):
    with rasterio.open(label_path) as src:
        data = src.read(1).ravel()
    row = {"SA": label_path.stem.replace("_labels", "")}
    for cid, cname in CLASS_NAMES.items():
        row[cname] = int(np.sum(data == cid))
    row["Background"] = int(np.sum(data == BACKGROUND))
    row["Total_valid"] = int(np.sum(data != BACKGROUND))
    rows.append(row)

dist_df = pd.DataFrame(rows).set_index("SA")

# Percentage view
pct_df = dist_df.drop(columns="Background").copy()
pct_df = pct_df.div(pct_df["Total_valid"], axis=0).multiply(100).round(1)
pct_df = pct_df.drop(columns="Total_valid")

print("Pixel counts per class per SA:")
display(dist_df)
print("\nClass proportions (% of valid pixels) per SA:")
display(pct_df)

Pixel counts per class per SA:


,Maize,Maize+Pumpkin,Beans+Maize,Cassava+Maize,Grass,Mixed,Background,Total_valid
SA,,,,,,,,
10085703,18573922,942586,0,0,0,0,113725227,19516508
10125706,2070139,2459444,0,0,47993004,870357,76060288,53392944
10145814,20430756,5303525,0,0,27550879,5470996,70258844,58756156
10165835,12084074,0,0,0,0,0,110506082,12084074
10165859,9961576,0,0,0,0,1423707,123211565,11385283
10185850,12779196,2330958,0,0,0,1046345,114433915,16156499
10305790,58736656,0,334996,254914,0,497776,61438158,59824342
10445787,10027122,1682978,0,0,0,6057220,110759820,17767320
10565691,54649,6814729,0,0,62600855,0,62514587,69470233



Class proportions (% of valid pixels) per SA:


,Maize,Maize+Pumpkin,Beans+Maize,Cassava+Maize,Grass,Mixed
SA,,,,,,
10085703,95.2,4.8,0.0,0.0,0.0,0.0
10125706,3.9,4.6,0.0,0.0,89.9,1.6
10145814,34.8,9.0,0.0,0.0,46.9,9.3
10165835,100.0,0.0,0.0,0.0,0.0,0.0
10165859,87.5,0.0,0.0,0.0,0.0,12.5
10185850,79.1,14.4,0.0,0.0,0.0,6.5
10305790,98.2,0.0,0.6,0.4,0.0,0.8
10445787,56.4,9.5,0.0,0.0,0.0,34.1
10565691,0.1,9.8,0.0,0.0,90.1,0.0
